# 🌾 Capstone Project: **Predicting Crop Yields**

**Time:** 6–10 hours (across the last days of Week 12)  **Prerequisites:** all of Phase 6  **Dataset:** `data/crop_yield.csv`: *new to you*

---

## 🧭 The scenario

You are the lead data scientist for a **farm-advisory service**. Each season, advisers visit thousands of smallholder fields and must decide where to send scarce agronomists, seed and fertiliser. A good **yield forecast per field** lets them:

* **triage**: find fields likely to under-perform *before* harvest,
* **recommend**: see which levers (water, nitrogen, variety) matter for each crop,
* **plan**: estimate regional supply.

Your job: build, validate, tune, ensemble, interpret and **defend** a yield model, and write the one-page summary the programme director will actually read.

> *This is the capstone: no step-by-step scaffolding like the Practical Assignment. You get a roadmap, a toolbox, checkpoints (✅) and rubric. The **decisions** are yours, and you must justify each one.*

## 📦 Deliverables
1. This notebook, completed top to bottom, reproducible (`SEED` everywhere).
2. A **written executive summary** (≈ 500 words) in the final cell: problem, approach, results with uncertainty, what drives yield, risks, recommendation.

## ⚖️ Rubric (100 points)

| area | pts | what we look for |
|---|---|---|
| **Framing & metric** | 10 | a metric that fits the business question and the data (crops differ ×10 in scale!); a justified baseline |
| **Data audit & leakage control** | 10 | missing data, skew, identifiers, anything that must not be a feature |
| **Validation design** | 15 | a split/CV that matches deployment; test set untouched until the end; paired folds |
| **Model benchmark** | 15 | ≥ 5 model families fairly compared with CV, mean ± spread, same folds |
| **Tuning** | 10 | principled search (Optuna ≥ 30 trials), sensible space, gain measured on CV |
| **Ensembling** | 10 | out-of-fold blending/stacking done without leakage; honest decision rule |
| **Final evaluation** | 10 | single test-set evaluation of a model chosen *without* the test set; per-crop errors |
| **Interpretation** | 10 | SHAP/permutation insights verified against agronomy; no causal over-claims |
| **Executive summary** | 10 | clear, quantified, honest about uncertainty & limits |
| **Bonus** | +5 | stretch challenges |

## 🗺️ Roadmap
M1 Frame & audit → M2 Metric & validation → M3 Benchmark → M4 Tune → M5 Ensemble → M6 Final test → M7 Interpret → M8 Report

## 📏 Ground rules
* The test set = the **last two seasons (2023–2024)**; it stays sealed until Milestone 6.
* All model *choices* come from **cross-validation on the training seasons**.
* Cite numbers from **your own runs** in the summary.

In [1]:
import sys; sys.path.append("../utils")          # lets us import the course helpers
from course_utils import *                       # load_housing(), check(), set_style(), ...
import numpy as np, pandas as pd, matplotlib.pyplot as plt

set_style()
np.random.seed(SEED)
from sklearn.model_selection import KFold, cross_validate, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
import xgboost as xgb, lightgbm as lgb, optuna, shap
import seaborn as sns
optuna.logging.set_verbosity(optuna.logging.WARNING)

## M1 · Frame the problem & audit the data

Read `data/DATA_DICTIONARY.md` (section *crop_yield.csv*), then explore. Pay attention to: the **scale of the target across crops**, **missing values**, **identifier columns**, and the relationship between **rainfall and yield**.

In [2]:
crops = load_crops()
print(crops.shape); display(crops.head(3))
print("\nmissing share:", crops.isna().mean()[lambda s: s > 0].round(3).to_dict())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.boxplot(data=crops, x="crop", y="yield_t_ha", ax=axes[0], color=PALETTE["blue"]); axes[0].set_title("Yield differs ×10 between crops")
axes[1].hist(np.log(crops["yield_t_ha"]), bins=50, color=PALETTE["orange"]); axes[1].set_title("log(yield) is much better behaved")
for crop_name, g in crops.groupby("crop"):
    axes[2].scatter(g["rainfall_mm"], g["yield_t_ha"] / g["yield_t_ha"].median(), s=5, alpha=.35, label=crop_name)
axes[2].set_xlabel("rainfall_mm"); axes[2].set_ylabel("yield / crop median"); axes[2].set_ylim(0, 4); axes[2].legend(markerscale=3, fontsize=8)
axes[2].set_title("Rainfall: hump-shaped (too dry AND too wet hurt)")
plt.tight_layout(); plt.show()

(2500, 19)


,field_id,year,region,crop,soil_type,seed_variety,altitude_m,rainfall_mm,avg_temp_c,humidity_pct,sunshine_hours,soil_ph,irrigated,nitrogen_kg_ha,phosphorus_kg_ha,planting_density,pest_pressure,field_size_ha,yield_t_ha
0,1,2015,Semi-arid,beans,volcanic,improved,1002.0,343.0,20.8,38.7,5.7,7.09,0,83.0,22.0,0.98,0.177,1.23,1.828
1,2,2024,Plateau,maize,loam,standard,1579.0,843.0,19.8,62.8,6.9,6.10,0,81.0,100.0,1.10,0.240,0.86,3.588
2,3,2017,Highlands,wheat,loam,hybrid,2216.0,781.0,12.1,81.0,5.2,5.14,0,134.0,32.0,0.97,0.101,0.49,1.764



missing share: {'humidity_pct': 0.044, 'soil_ph': 0.047, 'phosphorus_kg_ha': 0.064}


### 🧩 Problem 10.1 ⭐: Column roles

Define four lists that **partition** all 19 columns of `crops`:

* `target_col`: the name of the target (a string)
* `id_cols`: identifiers that must **never** be features
* `cat_cols`: categorical features (strings)
* `num_cols`: numeric features

*Think:* is `year` a feature? (Decide, and justify in a comment; the test set is the *future*.)

<details><summary>💡 Hint 1</summary>

Categoricals are the object columns: region, crop, soil_type, seed_variety.

</details>

<details><summary>💡 Hint 2</summary>

`field_id` is an identifier. `irrigated` is 0/1 (numeric is fine).

</details>

<details><summary>💡 Hint 3</summary>

Everything that is not the target, an id or a categorical is numeric (including `year` if you keep it, and `year` is *not* informative about yield in this data; either choice is defensible if justified).

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
target_col = TODO   # YOUR CODE HERE
id_cols = TODO      # YOUR CODE HERE
cat_cols = TODO     # YOUR CODE HERE
num_cols = TODO     # YOUR CODE HERE

# ---- self-check (don't edit) ----
def _ok():
    allc = set(crops.columns)
    feats = set(cat_cols) | set(num_cols)
    return (target_col == "yield_t_ha" and "field_id" in id_cols and set(cat_cols) == {"region", "crop", "soil_type", "seed_variety"}
            and not (feats & {target_col, "field_id"}) and feats | {target_col} | set(id_cols) | ({"year"} - feats) == allc
            and "rainfall_mm" in num_cols)
check("10.1", _ok, "Target + ids + features must cover every column (year may be dropped or kept); no id/target among the features.",
      needs=[target_col, id_cols, cat_cols, num_cols])

## M2 · Choose the metric and design the validation

### The metric
Yields run from ≈ 0.1 t/ha (beans, drought) to ≈ 30 t/ha (potato). **Plain RMSE in t/ha would be dominated by potato fields** and say almost nothing about beans. A natural fix for a multiplicative target is to model **`log(yield)`** and measure **RMSE on the log scale** (≈ RMSLE, "typical percentage error"): an error of 0.15 means *about 15% off*, for any crop.
We'll **train and score everything on `log(yield)`** and translate back to t/ha (and per-crop MAE) at the end for communication.

### The validation design
The service will forecast **next season's** fields. So: **train on seasons ≤ 2022, test on 2023–2024** (a time-forward holdout) and run *K-fold CV inside the training seasons* for all model choices (the data show no time trend, so random K-fold inside the training period is reasonable).

### 🧩 Problem 10.2 ⭐: Split and transform

Create:

* `train_df`, `test_df`: rows with `year <= 2022` / `year >= 2023`
* `X_train`, `X_test`: feature frames (`cat_cols + num_cols` only; **no ids, no target**)
* `y_train`, `y_test`: raw yields (t/ha) as Series
* `y_train_log`, `y_test_log`: `np.log` of those

<details><summary>💡 Hint 1</summary>

`train_df = crops[crops['year'] <= 2022]`; `test_df = crops[crops['year'] >= 2023]`.

</details>

<details><summary>💡 Hint 2</summary>

`features = cat_cols + num_cols`; `X_train = train_df[features]`.

</details>

<details><summary>💡 Hint 3</summary>

Reset nothing: keep the original index; `np.log(y_train)` returns a Series.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
train_df = TODO; test_df = TODO                      # YOUR CODE HERE
X_train = TODO; X_test = TODO                        # YOUR CODE HERE
y_train = TODO; y_test = TODO                        # YOUR CODE HERE
y_train_log = TODO; y_test_log = TODO                # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("10.2", lambda: len(X_train) + len(X_test) == len(crops) and 380 < len(X_test) < 620 and X_train.shape[1] == len(cat_cols) + len(num_cols)
      and train_df["year"].max() <= 2022 and test_df["year"].min() >= 2023 and np.allclose(y_train_log, np.log(y_train))
      and "field_id" not in X_train.columns and target_col not in X_train.columns,
      "Train = seasons ≤ 2022, test = 2023-24 (≈ 20% of rows); features only; logs of the raw yields.",
      needs=[X_train, X_test, y_train_log, y_test_log])

### 🧰 Your toolbox (provided)
A reusable CV scorer so every model is evaluated **on the same 5 folds** (paired comparison), on the log scale. `cv_score(model)` returns `(mean RMSE-log, std over folds, mean fit seconds)`.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)       # identical folds for every model

def cv_score(model, X=None, y=None):
    """5-fold CV RMSE on log(yield).  Returns (mean, std, mean fit seconds)."""
    X = X_train if X is None else X
    y = y_train_log if y is None else y
    res = cross_validate(model, X, y, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=1)
    return -res["test_score"].mean(), res["test_score"].std(ddof=1), res["fit_time"].mean()

prep = lambda: make_preprocessor(X_train, cat_cols=cat_cols, scale=True)    # impute + scale + one-hot, fitted inside each CV fold
print("toolbox ready: kf, cv_score(model), prep()")

## M3 · Benchmark ≥ 5 model families (plus a baseline)

Build a dict `candidates` of **pipelines** (each `make_pipeline(prep(), estimator)`) predicting `log(yield)`, including a **`"Baseline (mean)"`** (`DummyRegressor`) and at least **five real models** from different families, for example:
Ridge (linear) · Random Forest (bagging) · HistGradientBoosting · XGBoost · LightGBM. *(CatBoost is a great stretch addition; see the note in the stretch section about `clone`.)*

Then fill `cv_results`: a DataFrame indexed by model name with columns `rmse_mean`, `rmse_std`, `fit_s`, sorted by `rmse_mean`.

### 🧩 Problem 10.3 ⭐⭐: Fair benchmark

Build `candidates` and `cv_results` as described (use `cv_score`). Reasonable settings if you want a starting point:
Ridge(alpha=1) · RandomForest(300 trees, `max_features=0.5`) · HistGB(lr 0.05, 300 iters, depth 5) · XGBoost(500 trees, lr 0.05, depth 5, subsample 0.8, colsample 0.8) · LightGBM(500 trees, lr 0.05, 31 leaves, subsample 0.8 with `subsample_freq=1`).

<details><summary>💡 Hint 1</summary>

`candidates = {'Baseline (mean)': make_pipeline(prep(), DummyRegressor()), 'Ridge': make_pipeline(prep(), Ridge(1.0)), ...}`.

</details>

<details><summary>💡 Hint 2</summary>

Loop: `mean, std, fit = cv_score(model)`; collect `{'model': name, 'rmse_mean': mean, ...}`.

</details>

<details><summary>💡 Hint 3</summary>

`pd.DataFrame(rows).set_index('model').sort_values('rmse_mean')`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
candidates = TODO    # YOUR CODE HERE
cv_results = TODO    # YOUR CODE HERE

# ---- self-check (don't edit) ----
def _ok():
    base = [n for n in cv_results.index if "aseline" in n]
    real = [n for n in cv_results.index if "aseline" not in n]
    best = cv_results["rmse_mean"].min()
    return (len(base) == 1 and len(real) >= 5 and {"rmse_mean", "rmse_std", "fit_s"} <= set(cv_results.columns)
            and cv_results["rmse_mean"].is_monotonic_increasing and best < 0.25 * cv_results.loc[base[0], "rmse_mean"]
            and not cv_results.index[0].startswith(("Ridge", "Linear", "Baseline")))
check("10.3", _ok, "Baseline + ≥ 5 models, sorted by CV RMSE; the best model must beat the baseline by > 75% and must not be a linear model.",
      needs=[candidates, cv_results])

In [ ]:
cv_results.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
order = cv_results.drop(index=[n for n in cv_results.index if "aseline" in n]).index
ax.barh(order[::-1], cv_results.loc[order[::-1], "rmse_mean"], xerr=cv_results.loc[order[::-1], "rmse_std"], color=PALETTE["blue"], alpha=.85)
ax.set_xlabel("CV RMSE on log(yield)  (≈ typical % error): lower is better"); ax.set_title("Benchmark (5-fold CV, training seasons only)")
plt.tight_layout(); plt.show()

## M4 · Tune the most promising family with Optuna (≥ 30 trials)

Pick the best-looking **boosting** family and tune it with **Optuna** (TPE sampler, `seed=SEED`) against the 5-fold CV score. Think about the *search space* (log-scales, sensible ranges, Lesson 07's playbook): the number of trials is limited, so be deliberate.

### 🧩 Problem 10.4 ⭐⭐⭐: Principled tuning

* Write an Optuna objective that builds `make_pipeline(prep(), <your model>)` from `trial.suggest_*` values and returns `cv_score(model)[0]`.
* Run **≥ 30 trials** → `study`.
* `tuned_family`: the key of `cv_results` you tuned (e.g. `"LightGBM"`).
* `tuned_cv`: the best CV score found (`study.best_value`).
* `tuned_pipe`: an **unfitted** pipeline with `study.best_params` applied (so you can refit it later).

Your tuned CV score must beat the *default-settings* score of the same family.

<details><summary>💡 Hint 1</summary>

For LightGBM: tune `learning_rate` (log 0.01–0.2), `num_leaves` (8–64), `n_estimators` (200–800), `min_child_samples` (5–50), `subsample` (0.5–1), `colsample_bytree` (0.4–1), `reg_lambda` (log 1e-3–10); set `subsample_freq=1`.

</details>

<details><summary>💡 Hint 2</summary>

`study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED)); study.optimize(objective, n_trials=30)`.

</details>

<details><summary>💡 Hint 3</summary>

`tuned_pipe = make_pipeline(prep(), lgb.LGBMRegressor(**study.best_params, subsample_freq=1, random_state=SEED, verbose=-1))`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
def objective(trial):
    # YOUR CODE HERE
    return TODO

study = TODO          # YOUR CODE HERE
tuned_family = TODO   # YOUR CODE HERE
tuned_cv = TODO       # YOUR CODE HERE
tuned_pipe = TODO     # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("10.4", lambda: len(study.trials) >= 30 and tuned_family in cv_results.index and tuned_cv < cv_results.loc[tuned_family, "rmse_mean"]
      and abs(tuned_cv - study.best_value) < 1e-12 and hasattr(tuned_pipe, "fit"),
      "≥ 30 trials, and the tuned CV score must beat the default CV score of the family you tuned.", needs=[study, tuned_family, tuned_cv, tuned_pipe])

## M5 · Ensemble with **out-of-fold** predictions

Blend at least **three diverse models** (e.g. your tuned model, XGBoost, Random Forest, Ridge). Use **out-of-fold** predictions on the training seasons (Lesson 05) and a **non-negative linear meta-learner**.

### 🧩 Problem 10.5 ⭐⭐⭐: Leak-free blending

* `blend_names`: the list of ≥ 3 model names you blend (keys of a dict you build: `blend_models = {name: pipeline}`; include `tuned_pipe`)
* `oof`: DataFrame of **out-of-fold** predictions (`cross_val_predict(model, X_train, y_train_log, cv=kf)`), one column per model
* `meta`: `LinearRegression(positive=True)` fitted on `(oof, y_train_log)`; `weights`: `pd.Series(meta.coef_, index=blend_names)`
* `blend_oof_rmse`: RMSE of `meta.predict(oof)` vs `y_train_log`; `best_single_oof_rmse`: the lowest RMSE among the individual columns of `oof`

<details><summary>💡 Hint 1</summary>

`blend_models = {'Tuned': tuned_pipe, 'XGBoost': candidates['XGBoost'], 'Random Forest': candidates['Random Forest'], 'Ridge': candidates['Ridge']}`.

</details>

<details><summary>💡 Hint 2</summary>

`oof = pd.DataFrame({n: cross_val_predict(m, X_train, y_train_log, cv=kf) for n, m in blend_models.items()})`.

</details>

<details><summary>💡 Hint 3</summary>

Remember: the *meta-learner* is fitted on OOF predictions only. RMSE helper: `np.sqrt(np.mean((a - b) ** 2))`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
blend_models = TODO   # YOUR CODE HERE  (dict name -> pipeline)
blend_names = TODO    # YOUR CODE HERE
oof = TODO            # YOUR CODE HERE
meta = TODO           # YOUR CODE HERE
weights = TODO        # YOUR CODE HERE
blend_oof_rmse = TODO        # YOUR CODE HERE
best_single_oof_rmse = TODO  # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("10.5", lambda: len(blend_names) >= 3 and oof.shape == (len(X_train), len(blend_names)) and (weights.values >= -1e-12).all()
      and blend_oof_rmse <= best_single_oof_rmse + 1e-9 and abs(best_single_oof_rmse - min(np.sqrt(np.mean((y_train_log.values - oof[n].values) ** 2)) for n in oof)) < 1e-9,
      "≥ 3 models, OOF matrix (n_train × n_models), non-negative weights; the blend can't be worse than its best member on the OOF data.",
      needs=[blend_names, oof, weights, blend_oof_rmse, best_single_oof_rmse])

## M6 · The one-shot final evaluation

**Decision rule (set it *before* looking at the test set):** use the **blend** only if it beats the best single model's OOF RMSE by **at least 1%** (`blend_oof_rmse < 0.99 * best_single_oof_rmse`); otherwise ship the simpler tuned single model. Then refit on the full training seasons and evaluate on 2023–2024 **once**.

### 🧩 Problem 10.6 ⭐⭐⭐: Final test evaluation

* `final_choice`: `"blend"` or `"single"` according to the rule above
* `final_pred_log`: the test-set predictions of the chosen model on the **log scale** (for the blend: refit every blend model on the full training set, stack their test predictions as columns in the same order, and apply `meta.predict`; for the single model: refit `tuned_pipe`)
* `final_rmse_log`: RMSE on `y_test_log` (≈ "typical % error")
* `per_crop`: DataFrame indexed by crop with columns `mae_t_ha` (MAE in **t/ha**, after `np.exp`), `mean_yield` (mean actual yield of that crop in the test set) and `mae_pct` (= `100 * mae_t_ha / mean_yield`)

<details><summary>💡 Hint 1</summary>

`final_choice = 'blend' if blend_oof_rmse < 0.99 * best_single_oof_rmse else 'single'`.

</details>

<details><summary>💡 Hint 2</summary>

Blend: `Z = pd.DataFrame({n: m.fit(X_train, y_train_log).predict(X_test) for n, m in blend_models.items()})`; `final_pred_log = meta.predict(Z)`.

</details>

<details><summary>💡 Hint 3</summary>

Per crop: group the test rows by `X_test['crop']`; `np.abs(y_test - np.exp(final_pred_log))` is the absolute error in t/ha.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
final_choice = TODO      # YOUR CODE HERE
final_pred_log = TODO    # YOUR CODE HERE
final_rmse_log = TODO    # YOUR CODE HERE
per_crop = TODO          # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("10.6", lambda: final_choice in ("blend", "single") and final_choice == ("blend" if blend_oof_rmse < 0.99 * best_single_oof_rmse else "single")
      and len(final_pred_log) == len(y_test_log) and abs(final_rmse_log - np.sqrt(np.mean((y_test_log.values - np.asarray(final_pred_log)) ** 2))) < 1e-9
      and final_rmse_log < 0.22 and per_crop.shape[0] == 5 and {"mae_t_ha", "mean_yield", "mae_pct"} <= set(per_crop.columns),
      "Apply the decision rule; the test RMSE (log) should be well under 0.22; five crops in the table.",
      needs=[final_choice, final_pred_log, final_rmse_log, per_crop])

In [ ]:
per_crop.round(2)

## M7 · Interpret: does the model understand agronomy?

Fit a LightGBM with native categoricals (settings below) on the training seasons and explain its **test-set** predictions with SHAP (Lesson 08). Then check the model against two agronomic facts that the data were built to contain:

1. **Water is hump-shaped**: each crop has an optimal seasonal rainfall (maize ≈ 650 mm). Too dry *and* too wet reduce yield.
2. **Fertiliser has diminishing returns**, and only works when water is adequate.

### 🧩 Problem 10.7 ⭐⭐⭐: SHAP investigation

Train `shap_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=15, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1)` on `X_train` **with the four categorical columns converted to pandas `category` dtype** (convert `X_test` with the same categories) and `y_train_log`.

Compute SHAP values on the **test** features (`sv`), then store:
* `top5`: the five features with the largest mean |SHAP| (descending)
* `maize_best_rain`: among **maize** test rows, bin `rainfall_mm` into 150-mm-wide bins (`pd.cut` over `range(0, 1951, 150)`), take the mean SHAP of `rainfall_mm` in each bin, and store the **midpoint of the bin with the highest mean SHAP**

<details><summary>💡 Hint 1</summary>

`Xc_train = X_train.copy(); Xc_test = X_test.copy()`; for each cat column: `Xc_train[c] = Xc_train[c].astype('category'); Xc_test[c] = pd.Categorical(Xc_test[c], categories=Xc_train[c].cat.categories)`.

</details>

<details><summary>💡 Hint 2</summary>

`sv = shap.TreeExplainer(shap_model)(Xc_test)`; `sv.values` has shape (n_test, n_features) in the column order of `Xc_test`.

</details>

<details><summary>💡 Hint 3</summary>

`mask = (X_test['crop'] == 'maize').values`; `bins = pd.cut(X_test['rainfall_mm'][mask], bins=range(0, 1951, 150))`; `pd.Series(sv.values[mask, j]).groupby(bins.values).mean()`; `.idxmax().mid`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
shap_model = TODO       # YOUR CODE HERE (fitted)
top5 = TODO             # YOUR CODE HERE
maize_best_rain = TODO  # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("10.7", lambda: len(top5) == 5 and set(top5) <= set(X_test.columns) and "crop" in top5 and 300 <= maize_best_rain <= 750,
      "`crop` (base yield differs ×10) must be a top-5 driver; maize's best rainfall bin should be in the right ballpark of its optimum (300-750 mm).",
      needs=[top5, maize_best_rain])

In [ ]:
# Visual check of the planted structure (uses sv / cols from your SHAP solution)
cols = list(Xc_test.columns)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for crop_name, color in zip(["maize", "wheat", "potato"], [PALETTE["orange"], PALETTE["blue"], PALETTE["green"]]):
    m = (X_test["crop"] == crop_name).values
    axes[0].scatter(X_test["rainfall_mm"][m], sv.values[m, cols.index("rainfall_mm")], s=9, alpha=.6, label=crop_name, color=color)
axes[0].set_xlabel("rainfall_mm"); axes[0].set_ylabel("SHAP (log-yield)"); axes[0].set_title("Rainfall: crop-specific optimum (hump)"); axes[0].legend()
axes[1].scatter(X_test["nitrogen_kg_ha"], sv.values[:, cols.index("nitrogen_kg_ha")], s=9, alpha=.5, color=PALETTE["purple"]); axes[1].set_xlabel("nitrogen_kg_ha"); axes[1].set_title("Nitrogen: diminishing returns")
imp = pd.Series(np.abs(sv.values).mean(axis=0), index=cols).sort_values()
axes[2].barh(imp.index[-10:], imp.values[-10:], color=PALETTE["blue"]); axes[2].set_title("Mean |SHAP| (log-yield units)")
for ax in axes[:2]: ax.axhline(0, color="gray", lw=.6)
plt.tight_layout(); plt.show()

## M8 · The executive summary (≈ 500 words)

Write for the **programme director** (smart, not technical). Include, with *your* numbers:

1. **Problem & metric**: what we predict and what "error 0.15" means in plain words (≈ 15% off).
2. **Approach**: validation design (train ≤ 2022, test 2023–24, 5-fold CV inside), models compared, tuning, blending decision.
3. **Results with uncertainty**: baseline vs your model on the **test set**; per-crop errors in t/ha *and* %; how stable (CV spread).
4. **What drives yield**: 3 findings, each tied to an agronomic mechanism and flagged as *model-based, not causal*.
5. **Risks & limits**: where the model is weak (rare conditions, extrapolation beyond training ranges, a new region), data quality (missing soil/humidity), drift between seasons.
6. **Recommendation & next steps**: deploy / pilot? monitoring plan? what data would improve it?

> ✍️ **Your executive summary:** *(double-click and write here)*

## 🚀 Stretch challenges (+5 bonus for any one done well)

1. **CatBoost with native categoricals.** Benchmark it against your tuned model. ⚠️ *Gotcha:* scikit-learn's `cross_validate` cannot `clone` a `CatBoostRegressor` that carries `cat_features` in recent scikit-learn versions: write a manual K-fold loop (or pass categorical *indices* via `fit(..., cat_features=...)`).
2. **Segment robustness.** Evaluate error by region and by `seed_variety`. Is any group poorly served? Propose a fix.
3. **Prediction intervals.** Quantile LightGBM (α = 0.1, 0.9) on log-yield → 80% intervals; check coverage on 2023–24 and per crop.
4. **Classification twin.** Repeat the workflow on `data/loan_default.csv` (target `defaulted`): stratified CV, ROC-AUC/PR-AUC, calibrated probabilities, SHAP-based reason codes.
5. **Honest uncertainty.** Bootstrap the test set (1,000 resamples) to put a 95% interval on your final RMSE; compare it with the CV spread.

## ✅ Final checklist
- [ ] Metric justified for a target spanning crops with ×10 different scales
- [ ] No ids/target in features; test seasons untouched until M6
- [ ] Same folds for all models; mean ± std reported
- [ ] Tuning budget ≥ 30 trials with a sensible space; gain measured on CV
- [ ] Blend built from **out-of-fold** predictions; a pre-declared rule decided blend vs single
- [ ] Test set evaluated **once**, per-crop errors reported in t/ha and %
- [ ] Interpretations checked against domain facts; no causal claims
- [ ] Executive summary quantified, honest, readable

In [ ]:
scoreboard()